# Flatland TorchRL — Colab/Kaggle Notebook
Run `flatland_ppo_training_torchrl.py` and `scripts/puffer_sweep.py` with GPU.

**Notes**:
- This notebook assumes CUDA is available.
- It clones the full repo so local modules and configs resolve.
- For private repos, set `GITHUB_TOKEN` and (optionally) `FLATLAND_REPO` in the notebook environment.
- Logs go to `runs/` and sweep observations to `puffer_runs/`.


In [ ]:
# Clone repo
!git clone https://github.com/louiesmrs/flatland-torchrl.git

In [ ]:
# Remove any local virtualenv that can shadow system packages
!rm -rf .venv
!find . -name pyvenv.cfg -delete

In [ ]:
%cd /content/flatland-torchrl

In [ ]:
# System deps (needed for flatland_cutils build on Colab)
!apt-get update -y
!apt-get install -y build-essential

In [ ]:
# Install Python deps (CUDA)
!pip install -U pip
!pip install uv
!uv pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# Install repo + flatland_cutils
!uv pip  install -e .

In [ ]:
# Use non-editable install for flatland_cutils to avoid import shadowing in notebooks
!uv pip  install ./flatland_cutils

In [ ]:
!uv run python -c "import sys,runpy; \
sys.path.insert(0,'/content/flatland-torchrl'); \
sys.argv=['flatland_ppo_training_torchrl.py','--pretrained-network-path','model_checkpoints/flatland-rl__jiang_phase_1_3_7_to_10_agents__1__1769903932/flatland-rl__jiang_phase_1_3_7_to_10_agents__1__1769903932_4480000.tar','--num-envs','10','--num-steps','200','--vf-coef','0.13','--ent-coef','0.001','--max-grad-norm','0.2','--learning-rate','4.07e-5','--clip-coef','0.29','--seed','1','--exp-name','jiang_phase_1_3_7_to_10_agents','--curriculum-path','curriculums/jiang_phases_1_3_7_to_10_agents_30x30.json','--value-loss','l2']; \
runpy.run_path('/content/flatland-torchrl/flatland_ppo_training_torchrl.py', run_name='main')"

In [ ]:
!uv pip install -r tools/requirements.txt

In [ ]:
# Sweep run (force system site-packages before repo path)
!python -c "import sys,runpy; \
sys.path.insert(0,'/content/flatland-torchrl'); \
sys.argv=['puffer_sweep.py','--config','scripts/flatland_sweep.yaml','--use-gpu']; \
runpy.run_path('/content/flatland-torchrl/scripts/puffer_sweep.py', run_name='__main__')"

## TensorBoard
In Colab, run:
```
%load_ext tensorboard
%tensorboard --logdir runs
```

In [ ]:
# Bundle artifacts
!tar -czf artifacts.tgz runs puffer_runs

# Colab download helper
try:
    from google.colab import files

    files.download("artifacts.tgz")
except Exception:
    print("If running on Kaggle, find artifacts.tgz in /kaggle/working")